# Demo 2 — Bug-Triage Chain: Validators, Retry-with-Feedback, MLflow, Prompt Registry
## Session 6: Advanced Prompt Engineering

> **MLflow as the tape recorder.** Every prompt sent, every response returned, every tool call, every latency and token count — captured automatically by `mlflow.openai.autolog()` and the `@mlflow.trace` decorators below. Run the cells, then open the MLflow UI to replay every input and output the model saw.

**Goal:** Build a 3-step prompt chain (extract → classify → respond) that survives in production.

**What we ship:**
1. Three Pydantic schemas as typed contracts between steps
2. A `run_step` helper that validates output and **retries with the validator's complaint as feedback** (micro-Self-Refine)
3. MLflow tracing: one `CHAIN` parent span + three `LLM` child spans, one sub-span per retry attempt
4. Per-step prompts versioned in the **MLflow Prompt Registry** (`mlflow.genai.register_prompt`)
5. Per-step pass-rate analysis — find the weakest step programmatically
6. Prompt versioning + aliases for progressive rollout

**Why this matters:** A 4-step chain at 95% per step ships at 81%. At 80% per step it ships at 41%. The fix is **per-step validators + retry-with-feedback**, not better prompts.

**Before running:** configure via env vars (see Setup cell) — defaults work for local MLflow + OpenAI; falls back to mocks when `OPENAI_API_KEY` is unset.

Estimated cost when running with real OpenAI calls: ~$0.05 with `gpt-4o-mini`.

---

## Setup — env-var driven

The notebook boots from environment variables only — no hard-coded URLs, models, or keys.

```bash
export MLFLOW_TRACKING_URI=http://127.0.0.1:5000        # or Databricks workspace URL
export MLFLOW_EXPERIMENT=session6/demo_02_bug_triage    # optional override
export OPENAI_BASE_URL=https://api.openai.com/v1           # OpenAI-compatible endpoint
export OPENAI_API_KEY=sk-...                                # leave unset to use MOCK mode
export MODEL=gpt-4o-mini
```

`mlflow.openai.autolog()` turns every `chat.completions.create` call into a span with the full prompt, response, token counts, and latency — the tape recorder.

**Local-by-default.** Notebook ships with `OPENAI_BASE_URL=http://localhost:11434/v1` and `MODEL=qwen2.5-coder:7b` (Ollama). Set the env vars to point at OpenAI, Anthropic, vLLM, Together, etc. — any OpenAI-compatible endpoint.

**Cost tracking uses hypothetical local-LLM rates** (~$0.00005/1K tokens) so the cost-aware traces and scorers still produce meaningful numbers in the demos. Override the `PRICING` dict with your real per-1K rates for production.

In [ ]:
import os, json, time, textwrap
from typing import Callable, Any, Literal
from collections import defaultdict

import mlflow
from mlflow.entities import SpanType
from pydantic import BaseModel, Field, ValidationError
from openai import OpenAI

# ─── MLflow tracking ─────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI      = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
MLFLOW_TRACKING_USERNAME = os.getenv("MLFLOW_TRACKING_USERNAME")  # optional (Databricks/remote)
MLFLOW_TRACKING_PASSWORD = os.getenv("MLFLOW_TRACKING_PASSWORD")  # optional
MLFLOW_REGISTRY_URI      = os.getenv("MLFLOW_REGISTRY_URI", MLFLOW_TRACKING_URI)
MLFLOW_EXPERIMENT        = os.getenv("MLFLOW_EXPERIMENT", "session6/demo_02_bug_triage")

if MLFLOW_TRACKING_USERNAME:
    os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_TRACKING_USERNAME
if MLFLOW_TRACKING_PASSWORD:
    os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_TRACKING_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_REGISTRY_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

# ─── LLM (OpenAI-compatible: OpenAI, Azure, vLLM, Ollama, Together, etc.) ────
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")  # Ollama default
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "ollama")  # Ollama ignores; any non-empty string works
MODEL    = os.getenv("MODEL", "qwen2.5-coder:7b")  # local-default for chain demo

# Reproducibility default. Every LLM call below threads this through.
TEMPERATURE = float(os.getenv("TEMPERATURE", "0.0"))

# Per-1K-token pricing (USD). Update at each model release.
PRICING = {
    "gpt-4o":             {"in": 0.0025,   "out": 0.010},
    "gpt-4o-mini":        {"in": 0.00015,  "out": 0.0006},
    "o1-mini":            {"in": 0.003,    "out": 0.012},
    "claude-sonnet-4-6":  {"in": 0.003,    "out": 0.015},
    # Hypothetical local-LLM pricing (electricity + amortised GPU per 1K tokens — adjust to your infra)
    "qwen2.5-coder:14b":  {"in": 0.00005,  "out": 0.00005},
    "qwen2.5-coder:7b":   {"in": 0.00002,  "out": 0.00002},
    "llama3.1:8b":        {"in": 0.00002,  "out": 0.00002},
}

def price_of(model: str, usage) -> float:
    """USD cost for a single chat call. 0.0 in mock mode (no usage).
    Unknown models fall back to hypothetical local-LLM rates (~$0.00005/1K)."""
    if usage is None:
        return 0.0
    p = PRICING.get(model, {"in": 0.00005, "out": 0.00005})  # hypothetical fallback
    return usage.prompt_tokens / 1000 * p["in"] + usage.completion_tokens / 1000 * p["out"]


def tag_cost_latency(latency_ms: float, cost_usd: float) -> None:
    """Attach cost + latency to active span and current trace."""
    span = mlflow.get_current_active_span()
    if span:
        span.set_attribute("latency_ms", latency_ms)
        span.set_attribute("cost_usd", cost_usd)
    try:
        mlflow.update_current_trace(tags={
            "cost_usd":   f"{cost_usd:.6f}",
            "latency_ms": f"{latency_ms:.0f}",
        })
    except Exception:
        pass


client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY or "no-key")

# ─── Turn MLflow into the tape recorder ──────────────────────────────────────
# Captures every chat.completions.create call — request, response, tokens, latency, cost.
mlflow.openai.autolog()

USE_MOCK = not OPENAI_API_KEY
print(f"MLflow:  {MLFLOW_TRACKING_URI}  (exp: {MLFLOW_EXPERIMENT})")
print(f"LLM:     {OPENAI_BASE_URL}  model={MODEL}  mock={USE_MOCK}  temperature={TEMPERATURE}")
print(f"UI:      open {MLFLOW_TRACKING_URI}  → Experiments → {MLFLOW_EXPERIMENT}")
print(f"NOTE: defaults to local Ollama (http://localhost:11434/v1). Set OPENAI_BASE_URL/OPENAI_API_KEY/MODEL to point at OpenAI/Anthropic/vLLM.")

## The accuracy-floor problem

Four LLM calls in series, each independently ~good:

| Step | Per-step pass-rate |
|------|--------------------|
| Extract bug signal | 0.95 |
| Classify component | 0.92 |
| Find duplicates | 0.88 |
| Draft reply | 0.80 |
| **End-to-end** | **0.95 × 0.92 × 0.88 × 0.80 ≈ 0.62** |

Your pipeline ships at 62%. The weakest step (0.80) is dragging everything.

**The recovery move:** put a typed validator on every step's output, and re-prompt with the validator's complaint when it fails. That's what this notebook builds.

## Step 1 — Pydantic schemas (the typed contracts)

Three schemas, one per chain step. Each schema is what the *next* step receives. `Literal[...]` enums make malformed enums a `ValidationError` we can catch and retry.

In [ ]:
class ExtractedSignal(BaseModel):
    symptom: str       = Field(..., description="One-line description of what is going wrong.")
    surface: str       = Field(..., description="Affected surface: route, screen, API, etc.")
    time_seen: str     = Field(..., description="When the issue was first seen.")
    env: str           = Field(..., description="Environment: prod, staging, dev.")
    freq: str          = Field(..., description="Frequency: every time, intermittent, one-off.")


class Classification(BaseModel):
    severity:   Literal["critical", "high", "medium", "low"]
    component:  str
    confidence: Literal["high", "medium", "low"]


class TriageResponse(BaseModel):
    severity:    Literal["critical", "high", "medium", "low"]
    component:   str
    summary:     str
    next_action: Literal["page", "queue", "close", "ask"]


print("Schemas defined:")
for s in (ExtractedSignal, Classification, TriageResponse):
    print(f"  - {s.__name__}: {list(s.model_fields.keys())}")

## Step 2 — Register the three prompts in MLflow Prompt Registry

Each step gets its own registered prompt with a Jinja-style `{{ slot }}` for inputs **and** an optional `{{ validator_feedback }}` slot used on retries. Tags make them searchable in the Registry UI.

In [ ]:
EXTRACT_TEMPLATE = '''You are a bug triage assistant. Extract structured signal from a bug report.
Return STRICT JSON matching the ExtractedSignal schema. No prose.

Required fields: symptom, surface, time_seen, env, freq.

{{ validator_feedback }}

Bug report:
{{ raw_report }}'''

CLASSIFY_TEMPLATE = '''You are a bug triage assistant. Given an extracted signal, classify severity and component.

Severity rubric:
- critical: data loss, security breach, outage
- high:     major feature broken, no workaround
- medium:   feature partially broken, workaround exists
- low:      cosmetic / minor inconvenience

Confidence: high | medium | low.
Component: short string (e.g. "auth", "checkout", "dashboard", "reporting", "infra").

{{ validator_feedback }}

Extracted signal:
{{ signal }}

Return STRICT JSON matching the Classification schema. No prose.'''

RESPOND_TEMPLATE = '''You are an on-call engineer drafting the first reply on a triage ticket.
Tone: factual, blameless, action-oriented.

next_action rubric:
- page:  critical — wake someone up
- queue: high/medium — schedule into sprint
- close: not a bug / low
- ask:   need more info from reporter

{{ validator_feedback }}

Extracted signal:
{{ signal }}

Classification:
{{ classification }}

Return STRICT JSON matching the TriageResponse schema. No prose.'''

pv_extract = mlflow.genai.register_prompt(
    name="triage_extract",
    template=EXTRACT_TEMPLATE,
    commit_message="initial — extract bug signal with validator_feedback slot",
    tags={"chain": "bug_triage", "step": "1_extract", "schema": "ExtractedSignal"},
)
pv_classify = mlflow.genai.register_prompt(
    name="triage_classify",
    template=CLASSIFY_TEMPLATE,
    commit_message="initial — severity rubric + Literal enums",
    tags={"chain": "bug_triage", "step": "2_classify", "schema": "Classification"},
)
pv_respond = mlflow.genai.register_prompt(
    name="triage_respond",
    template=RESPOND_TEMPLATE,
    commit_message="initial — blameless tone, 4-way next_action",
    tags={"chain": "bug_triage", "step": "3_respond", "schema": "TriageResponse"},
)

print(f"triage_extract  registered as version {pv_extract.version}")
print(f"triage_classify registered as version {pv_classify.version}")
print(f"triage_respond  registered as version {pv_respond.version}")

EXTRACT_PROMPT  = mlflow.genai.load_prompt(f"prompts:/triage_extract/{pv_extract.version}")
CLASSIFY_PROMPT = mlflow.genai.load_prompt(f"prompts:/triage_classify/{pv_classify.version}")
RESPOND_PROMPT  = mlflow.genai.load_prompt(f"prompts:/triage_respond/{pv_respond.version}")
print("\nAll three prompts loaded back from the Registry.")

## Step 3 — The `run_step` helper: validator + retry-with-feedback

This is the heart of the chain. Each attempt is its own MLflow span so the retry pattern is **visible in the trace tree**. On `ValidationError`, the error string is stuffed into `ctx["_<name>_feedback"]` and the prompt's `{{ validator_feedback }}` slot picks it up next iteration. That's a single-round Self-Refine embedded in a chain step.

In [ ]:
class ChainStepError(Exception):
    def __init__(self, step_name: str, last_output: Any, last_error: str):
        self.step_name  = step_name
        self.last_output = last_output
        self.last_error = last_error
        super().__init__(f"{step_name}: {last_error}")


# Attempts counter — we'll read this in Section 12 for per-step pass-rate analysis.
ATTEMPT_LOG: dict[str, list[int]] = defaultdict(list)


def run_step(
    name: str,
    fn: Callable[[dict], str],
    ctx: dict,
    schema: type[BaseModel],
    retries: int = 2,
) -> dict:
    """Run one chain step with Pydantic validator + retry-with-feedback.

    Each attempt opens its own LLM span with attributes:
        attempt:        0, 1, 2 ...
        has_feedback:   bool (True on retries)
        validator_pass: True on success, False on schema error
    """
    last_output, last_error = None, None
    for attempt in range(retries + 1):
        with mlflow.start_span(
            name=f"{name}_attempt_{attempt}",
            span_type=SpanType.LLM,
        ) as sp:
            sp.set_attribute("attempt", attempt)
            sp.set_attribute("has_feedback", f"_{name}_feedback" in ctx)
            # Tape-recorder: explicit input snapshot for this attempt.
            sp.set_inputs({
                "prompt": f"{name} attempt {attempt}",
                "params": {
                    "schema": schema.__name__,
                    "retries_remaining": retries - attempt,
                    "validator_feedback": ctx.get(f"_{name}_feedback", ""),
                    "context_keys": list(ctx.keys()),
                },
            })

            raw = fn(ctx)
            last_output = raw

            try:
                parsed = schema.model_validate_json(raw)
                ctx[name] = parsed.model_dump()
                ctx.pop(f"_{name}_feedback", None)
                sp.set_attribute("validator_pass", True)
                # Tape-recorder: what came back + the parsed payload.
                sp.set_outputs({"text": raw, "parsed": ctx[name]})
                ATTEMPT_LOG[name].append(attempt + 1)
                return ctx
            except ValidationError as e:
                last_error = str(e)
                sp.set_attribute("validator_pass", False)
                sp.set_attribute("validator_error", last_error[:500])
                # Tape-recorder: capture the malformed response so retry is auditable.
                sp.set_outputs({"text": raw, "validator_error": last_error[:500]})
                ctx[f"_{name}_feedback"] = (
                    f"Your previous output failed validation on attempt {attempt+1}:\n"
                    f"{last_error}\n\nReturn STRICT JSON matching the schema. No prose."
                )

    ATTEMPT_LOG[name].append(retries + 1)
    raise ChainStepError(name, last_output=last_output, last_error=last_error or "")


print("run_step ready — validates output with Pydantic, retries with feedback on failure.")

## Step 4 — The three step functions

Each step:
1. is decorated with `@mlflow.trace` as an `LLM` span (so it nests under the chain)
2. loads its prompt from the Registry, formats with current `ctx`
3. calls the model with `response_format={"type":"json_object"}`
4. returns raw JSON string for `run_step` to validate

When `USE_MOCK` is true, we use deterministic responses that occasionally return malformed JSON on first attempt so the retry path is visible.

In [ ]:
# A toggle the mocks consult so we can show a retry happening at least once.
MOCK_FAIL_FIRST = {"extract": True, "classify": True, "respond": False}


def _mock(step: str, ctx: dict) -> str:
    """Return deterministic mock JSON. On first attempt of `extract` and `classify`,
    return a malformed payload so retry-with-feedback fires once."""
    has_fb = f"_{step}_feedback" in ctx
    if MOCK_FAIL_FIRST[step] and not has_fb:
        # Return invalid JSON (missing required field / bad enum) to trip the validator.
        if step == "extract":
            return json.dumps({"symptom": "unknown"})  # missing many required fields
        if step == "classify":
            return json.dumps({"severity": "URGENT", "component": "x", "confidence": "high"})
    # Valid responses keyed off the bug report text.
    report = ctx.get("raw_report", "").lower()
    if step == "extract":
        return json.dumps({
            "symptom":  "sql injection in /login" if "sql" in report else "slow page load",
            "surface":  "/login"                  if "sql" in report else "/dashboard",
            "time_seen": "today",
            "env":       "prod",
            "freq":      "every time" if "every" in report else "intermittent",
        })
    sig = ctx.get("extract", {})
    if step == "classify":
        sev = "critical" if "sql" in sig.get("symptom", "").lower() else (
              "high"     if "slow" in sig.get("symptom", "").lower() else "medium")
        return json.dumps({"severity": sev, "component": sig.get("surface", "unknown"),
                            "confidence": "high"})
    # respond
    cls = ctx.get("classify", {})
    nx = {"critical": "page", "high": "queue", "medium": "queue", "low": "close"}[cls["severity"]]
    return json.dumps({
        "severity":    cls["severity"],
        "component":   cls["component"],
        "summary":     f"{sig.get('symptom', 'issue')} on {sig.get('surface', 'unknown')} in {sig.get('env', 'prod')}",
        "next_action": nx,
    })


def _llm(rendered: str) -> tuple[str, float, float]:
    """Call the configured LLM with JSON mode. Returns (text, latency_ms, cost_usd)."""
    t0 = time.time()
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": rendered}],
        response_format={"type": "json_object"},
        temperature=TEMPERATURE,   # 0.0 by default: deterministic structured output
        max_tokens=400,
    )
    latency_ms = (time.time() - t0) * 1000
    cost = price_of(MODEL, resp.usage)
    return resp.choices[0].message.content, latency_ms, cost


def _run_step_llm(step_name: str, rendered: str) -> str:
    """Shared wrapper: mock-or-LLM, then tag cost/latency on the active span + trace."""
    if USE_MOCK:
        t0 = time.time()
        # ctx isn't visible here — _mock callers handle their own ctx; we only land here from steps
        # that already constructed `rendered`. The mock branch in step functions handles this.
        raw = ""
        latency_ms = (time.time() - t0) * 1000
        cost = 0.0
    else:
        raw, latency_ms, cost = _llm(rendered)
    tag_cost_latency(latency_ms, cost)
    return raw


@mlflow.trace(name="extract_signal", span_type=SpanType.LLM,
              attributes={"prompt": "triage_extract", "schema": "ExtractedSignal"})
def extract_signal(ctx: dict) -> str:
    rendered = EXTRACT_PROMPT.format(
        raw_report=ctx["raw_report"],
        validator_feedback=ctx.get("_extract_feedback", ""),
    )
    # Tape-recorder: explicit prompt + params on this LLM span.
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": rendered, "params": {"model": MODEL, "prompt_name": "triage_extract",
                                                          "temperature": TEMPERATURE}})
    if USE_MOCK:
        raw = _mock("extract", ctx)
        tag_cost_latency(0.0, 0.0)
    else:
        raw, latency_ms, cost = _llm(rendered)
        tag_cost_latency(latency_ms, cost)
    if span:
        span.set_outputs({"text": raw})
    return raw


@mlflow.trace(name="classify_component", span_type=SpanType.LLM,
              attributes={"prompt": "triage_classify", "schema": "Classification"})
def classify_component(ctx: dict) -> str:
    rendered = CLASSIFY_PROMPT.format(
        signal=json.dumps(ctx["extract"], indent=2),
        validator_feedback=ctx.get("_classify_feedback", ""),
    )
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": rendered, "params": {"model": MODEL, "prompt_name": "triage_classify",
                                                          "temperature": TEMPERATURE}})
    if USE_MOCK:
        raw = _mock("classify", ctx)
        tag_cost_latency(0.0, 0.0)
    else:
        raw, latency_ms, cost = _llm(rendered)
        tag_cost_latency(latency_ms, cost)
    if span:
        span.set_outputs({"text": raw})
    return raw


@mlflow.trace(name="draft_response", span_type=SpanType.LLM,
              attributes={"prompt": "triage_respond", "schema": "TriageResponse"})
def draft_response(ctx: dict) -> str:
    rendered = RESPOND_PROMPT.format(
        signal=json.dumps(ctx["extract"], indent=2),
        classification=json.dumps(ctx["classify"], indent=2),
        validator_feedback=ctx.get("_respond_feedback", ""),
    )
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": rendered, "params": {"model": MODEL, "prompt_name": "triage_respond",
                                                          "temperature": TEMPERATURE}})
    if USE_MOCK:
        raw = _mock("respond", ctx)
        tag_cost_latency(0.0, 0.0)
    else:
        raw, latency_ms, cost = _llm(rendered)
        tag_cost_latency(latency_ms, cost)
    if span:
        span.set_outputs({"text": raw})
    return raw


print("3 step functions ready: extract_signal → classify_component → draft_response")

## Step 5 — The chain

One `@mlflow.trace` outer function with `span_type=SpanType.CHAIN`. It calls the three steps through `run_step` and tags the trace with the classification + the **pinned prompt versions** so we can search later.

In [ ]:
CHAIN_VERSION = "v1.0"


@mlflow.trace(name="bug_triage", span_type=SpanType.CHAIN,
              attributes={"chain_version": CHAIN_VERSION})
def triage(raw_report: str) -> TriageResponse:
    # Tape-recorder: the chain's input — the original bug report.
    span = mlflow.get_current_active_span()
    if span:
        span.set_inputs({"prompt": raw_report,
                         "params": {"model": MODEL, "chain_version": CHAIN_VERSION}})

    ctx = {"raw_report": raw_report}
    ctx = run_step("extract",  extract_signal,     ctx, schema=ExtractedSignal)
    ctx = run_step("classify", classify_component, ctx, schema=Classification)
    ctx = run_step("respond",  draft_response,     ctx, schema=TriageResponse)

    mlflow.update_current_trace(tags={
        "chain_version":    CHAIN_VERSION,
        "severity":         ctx["classify"]["severity"],
        "component":        ctx["classify"]["component"],
        "prompt_extract":   f"triage_extract/{EXTRACT_PROMPT.version}",
        "prompt_classify": f"triage_classify/{CLASSIFY_PROMPT.version}",
        "prompt_respond":  f"triage_respond/{RESPOND_PROMPT.version}",
    })
    result = TriageResponse.model_validate(ctx["respond"])
    # Tape-recorder: the chain's final structured output.
    if span:
        span.set_outputs({"text": result.model_dump_json(), "parsed": result.model_dump()})
    return result


print(f"Chain '{CHAIN_VERSION}' ready. Calling triage() will produce one CHAIN span with 3 LLM children.")

## Step 6 — Run on five real bug reports

Diverse mix: a critical security issue, a perf regression, a flaky test, a deprecation notice, a UX nit. Watch how the chain assigns different `next_action` values.

In [ ]:
REPORTS = [
    "Title: SQL injection in /login form\n"
    "Tester pasted ' OR 1=1 -- into the email field every time and got logged in as admin. Prod.",

    "Title: Dashboard takes 45s to load\n"
    "After the v2.3 release the /dashboard route shows a spinner for ~45 seconds before rendering. Affects every prod user.",

    "Title: Flaky test test_payment_retry intermittently failing\n"
    "CI run fails ~1 in 8 builds. Local repro intermittent. Staging only.",

    "Title: Deprecation warning from pandas in reporting job\n"
    "Nightly reporting prints FutureWarning about .append() being removed. Output still correct. Prod.",

    "Title: Button label says 'Sumbit' not 'Submit'\n"
    "Typo on the contact form button. Cosmetic only. Prod.",
]

responses: list[TriageResponse] = []
for i, report in enumerate(REPORTS, 1):
    print(f"\n{'='*70}")
    print(f"Report {i}: {report.splitlines()[0]}")
    print("-" * 70)
    resp = triage(report)
    responses.append(resp)
    print(f"  severity    : {resp.severity}")
    print(f"  component   : {resp.component}")
    print(f"  summary     : {resp.summary}")
    print(f"  next_action : {resp.next_action}")

print(f"\n{'='*70}\nAll {len(responses)} reports triaged. Each produced one CHAIN trace + 3 LLM children.")

## Step 7 — Open the MLflow UI

**→ http://127.0.0.1:5000 → Experiments → `session6/demo_02_bug_triage` → Traces tab**

Click into any `bug_triage` trace. You should see:
```
bug_triage                                  (CHAIN, our outer @mlflow.trace)
  ├─ extract_signal                          (LLM)
  │    ├─ extract_attempt_0                  (LLM, validator_pass=False on mock)
  │    └─ extract_attempt_1                  (LLM, validator_pass=True)
  ├─ classify_component                      (LLM)
  │    ├─ classify_attempt_0                 (validator_pass=False on mock)
  │    └─ classify_attempt_1                 (validator_pass=True)
  └─ draft_response                          (LLM, single attempt)
       └─ respond_attempt_0                  (validator_pass=True)
```

Click `extract_attempt_0` to see the failing JSON and the `validator_error` attribute. Click `extract_attempt_1` to see how the next call had `has_feedback=True` and recovered.

## Step 8 — Search traces programmatically

Tags are first-class search filters. Pull all critical-severity classifications across this run.

In [ ]:
import pandas as pd

df = mlflow.search_traces(
    experiment_names=[MLFLOW_EXPERIMENT],
    filter_string=f"tags.chain_version = '{CHAIN_VERSION}' AND tags.severity = 'critical'",
    max_results=10,
)

if df.empty:
    print("No critical-severity traces yet — are tags populated? Refresh the MLflow UI.")
else:
    cols = [c for c in ["trace_id", "tags.severity", "tags.component",
                         "tags.prompt_classify", "execution_time_ms"] if c in df.columns]
    print(f"Found {len(df)} critical traces:")
    pd.set_option("display.max_colwidth", 60)
    print(df[cols].to_string(index=False) if cols else df.head().to_string())

## Step 9 — Per-step pass-rate analysis

Tally how many attempts each step needed across all 5 reports. **The weakest step is your prompt-engineering target** — always go after the lowest bar, not the one you remember last touching.

### Mapping back to the slide intuition

The presenter's accuracy-floor slide showed:

```
Step           Per-step pass-rate
extract            0.95
classify           0.92
duplicates         0.88
respond            0.80
End-to-end:    0.95 × 0.92 × 0.88 × 0.80 ≈ 0.62  (62%)
```

What you'll see below is the **same math, measured from this notebook's traces**: the per-step first-try pass-rate is the local equivalent of those slide numbers. Multiply them together → predicted end-to-end pass-rate. Fix the **weakest step** (e.g. lift `respond` from 0.80 → 0.92) and the product jumps disproportionately — that's why per-step validators + retry-with-feedback matter more than tweaking the strongest step's prompt.

In [ ]:
print(f"{'step':<10} {'runs':<6} {'1st-try pass':<14} {'avg attempts':<14}")
print("-" * 50)
weakest = (None, 1.0)
for step, attempts in ATTEMPT_LOG.items():
    runs       = len(attempts)
    first_pass = sum(1 for a in attempts if a == 1) / runs
    avg        = sum(attempts) / runs
    print(f"{step:<10} {runs:<6} {first_pass:<14.2%} {avg:<14.2f}")
    if first_pass < weakest[1]:
        weakest = (step, first_pass)

print()
if weakest[0]:
    print(f"Weakest step: '{weakest[0]}' (first-try pass rate {weakest[1]:.0%}). Tune that prompt next.")
else:
    print("All steps passed on first try — nothing to tune.")

### Same analysis, but read from MLflow traces (the production pattern)

The in-memory `ATTEMPT_LOG` is fine for this notebook. In production, the chain runs across many services and pods — you cannot rely on a Python global. The traces themselves are the source of truth.

The cell below uses `mlflow.search_traces` to pull every `bug_triage` trace, walks the span tree per trace, and tallies `validator_pass` outcomes per step. This is what you would run from a notebook or a daily CI job to monitor pass-rate drift.

In [ ]:
# -- Per-step pass-rate, computed from traces (the production pattern) -----
#
# Walks every recent `bug_triage` trace, finds spans named `<step>_attempt_<N>`,
# and tallies validator_pass=True over total attempts per step.

from collections import defaultdict as _dd

STEPS = ["extract", "classify", "respond"]
attempts_by_step:  dict[str, int] = _dd(int)
passes_by_step:    dict[str, int] = _dd(int)

try:
    traces_df = mlflow.search_traces(
        experiment_names=[MLFLOW_EXPERIMENT],
        filter_string=f"tags.chain_version = '{CHAIN_VERSION}'",
        max_results=50,
    )
except Exception as exc:
    print(f"search_traces failed ({exc}); skipping trace-derived pass-rate.")
    traces_df = None

if traces_df is not None and not traces_df.empty:
    for _, row in traces_df.iterrows():
        trace = row.get("trace") if "trace" in row else None
        if trace is None:
            # Older MLflow surface: fetch by trace_id
            try:
                trace = mlflow.get_trace(row["trace_id"])
            except Exception:
                continue
        spans = getattr(trace, "data", trace).spans if hasattr(trace, "data") else trace.spans
        for sp in spans:
            name = getattr(sp, "name", "")
            # Match `<step>_attempt_<N>`
            for step in STEPS:
                if name.startswith(f"{step}_attempt_"):
                    attempts_by_step[step] += 1
                    attrs = getattr(sp, "attributes", {}) or {}
                    if attrs.get("validator_pass") is True or str(attrs.get("validator_pass")).lower() == "true":
                        passes_by_step[step] += 1
                    break

    print(f"{'Step':<20} {'Pass-rate':>10} {'Attempts':>12}")
    print("-" * 46)
    product = 1.0
    weakest = (None, 1.0)
    for step in STEPS:
        attempts = attempts_by_step[step]
        passes   = passes_by_step[step]
        rate     = (passes / attempts) if attempts else 0.0
        product *= rate if attempts else 1.0
        label = {
            "extract":  "extract_signal",
            "classify": "classify_component",
            "respond":  "draft_response",
        }[step]
        marker = ""
        if attempts and rate < weakest[1]:
            weakest = (label, rate)
        print(f"{label:<20} {rate:>9.0%}  {passes:>5}/{attempts:<5}")
    if weakest[0]:
        # Highlight weakest
        print()
        print(f"Weakest step: {weakest[0]}  ({weakest[1]:.0%})   ← target for the next prompt edit")

    print()
    print(f"End-to-end (product of first-try rates) ≈ {product:.4f}  ≈ {product*100:.2f}%")
    if weakest[0]:
        # Quick what-if: if weakest step jumps to 0.92, recompute
        target = 0.92
        new_product = 1.0
        for step in STEPS:
            attempts = attempts_by_step[step]
            if not attempts:
                continue
            r = (passes_by_step[step] / attempts)
            label = {"extract": "extract_signal", "classify": "classify_component", "respond": "draft_response"}[step]
            new_product *= (target if label == weakest[0] else r)
        print(f"What-if: lift {weakest[0]} to {target:.2f}  →  end-to-end ≈ {new_product:.4f}  ≈ {new_product*100:.2f}%")
else:
    print("No traces found yet. Run the chain on a few reports first.")

## Step 10 — Versioning: register `triage_classify` v2

Suppose pass-rate analysis points at classify. We tweak the prompt — add an explicit confidence-calibration sentence — and register **v2** of *just that prompt*. The chain code doesn't change; only the loaded version does.

In [ ]:
CLASSIFY_TEMPLATE_V2 = CLASSIFY_TEMPLATE + (
    "\n\nConfidence calibration:\n"
    "- Use 'high' only when the signal explicitly names the severity-determining evidence.\n"
    "- Default to 'medium' when inferring severity from indirect cues.\n"
    "- Use 'low' when the signal is ambiguous."
)

pv_classify_v2 = mlflow.genai.register_prompt(
    name="triage_classify",
    template=CLASSIFY_TEMPLATE_V2,
    commit_message="v2 — confidence calibration: 'high' only with explicit evidence",
    tags={"chain": "bug_triage", "step": "2_classify", "schema": "Classification"},
)
print(f"triage_classify now has versions 1 and {pv_classify_v2.version}.")
print("\nTo swap, the chain just loads a different URI:")
print(f"  mlflow.genai.load_prompt('prompts:/triage_classify/{pv_classify_v2.version}')")

## Step 11 — Aliases for progressive rollout

Pinning by version (`prompts:/triage_classify/1`) is deterministic but rigid. **Aliases** let you point a name like `@production` at any version, and rotate it without changing chain code.

Typical rollout:
1. v1 → `@production`, v2 → `@candidate`
2. Run candidate on 10% of traffic, compare `tags.severity` distributions
3. Promote: set `@production` to v2

In [ ]:
try:
    mlflow.genai.set_prompt_alias("triage_classify", alias="production", version=1)
    mlflow.genai.set_prompt_alias("triage_classify", alias="candidate",  version=pv_classify_v2.version)

    prod_prompt = mlflow.genai.load_prompt("prompts:/triage_classify@production")
    cand_prompt = mlflow.genai.load_prompt("prompts:/triage_classify@candidate")
    print(f"@production → version {prod_prompt.version}")
    print(f"@candidate  → version {cand_prompt.version}")
    print("\nPromote candidate to production:")
    print("  mlflow.genai.set_prompt_alias('triage_classify', alias='production',")
    print(f"                                version={pv_classify_v2.version})")
except Exception as e:
    print(f"Alias API not available in this MLflow version: {e}")
    print("Fallback: pin by version number explicitly when loading.")

## Takeaways

1. **Every step needs a validator.** A 4-step chain at 95% per step ships at 81%; without typed contracts you don't even know which step degraded.
2. **Retry-with-feedback is micro-Self-Refine inside the chain.** The validator's error string becomes the next call's correction prompt — most JSON / enum failures recover on attempt 2.
3. **Per-step pass-rate analysis finds the weakest step.** Always tune the lowest bar, not the one you last touched. MLflow trace tags + `search_traces` make this a 3-line query.
4. **The Prompt Registry makes per-step versioning trivial.** Each step has its own version history; aliases (`@production`, `@candidate`) enable progressive rollout without chain code changes.

### What to try next
- Replace `_mock` with real OpenAI calls (set `OPENAI_API_KEY`) and watch real pass-rates emerge.
- Add a 4th step (`find_duplicates`) and watch the accuracy floor drop — then prove validators recover it.
- Swap `triage_classify@production` to v2 and re-run; compare severity distributions across `chain_version` tags.

## ⚠️ Reasoning model caveat

This notebook assumes a **non-reasoning** base model. On `o1` / `o3` / Claude Extended Thinking / Gemini Thinking:
- Drop the CoT / ToT / Plan-and-Solve scaffolding — the model does it internally
- Use `developer` role instead of `system` on `o1-2024-12-17+`
- Never pass OpenAI `reasoning_effort` and Gemini `thinking_level` through the same OpenAI-compatible adapter
- See Block 5 of the slides for what's redundant vs what survives

Chain-with-validators (extract → classify → respond) is **still useful on reasoning models** — the per-step typed contracts and retry-with-feedback are independent of the reasoning trace. Only the in-prompt CoT scaffolding becomes redundant.

## 🎥 Replay every input + output in MLflow

Every prompt and every response from this demo is now in MLflow. Open the UI and click any trace to see the full I/O log.

In [ ]:
import urllib.parse
import IPython.display as D

ui = MLFLOW_TRACKING_URI.rstrip("/")
exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
exp_id = exp.experiment_id if exp else "0"
link = f"{ui}/#/experiments/{exp_id}?searchFilter=&orderByKey=attributes.start_time&orderByAsc=false"
D.display(D.Markdown(f"**Open the experiment in MLflow:** [{link}]({link})"))

# Recent traces (last 5)
recent = mlflow.search_traces(experiment_ids=[exp_id], max_results=5)
recent[["request_id", "timestamp_ms", "execution_time_ms", "tags"]] if not recent.empty else "No traces yet — run the cells above."